# 第 14 课：真实开源音频上的最小 CTC 训练实验

我们使用 Free Spoken Digit Dataset 的真实语音，构造一个很小的可过拟合实验。重点是看懂完整数据流，不追求泛化性能。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | CTC 核心 |
| 建议投入 | 4～6 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 13 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | 真实音频训练、CTC spike、过拟合实验 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：真实音频训练、CTC spike、过拟合实验。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：编码器输出时间长度；概率与 log 概率；有效帧 mask；重复 token 与 blank 的区别。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](13_CTC_Greedy与PrefixBeamSearch.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：logits/log-probs、input_lengths、targets、target_lengths
  ↓ 本课要学会的变换、状态或判断
输出：可验算的 CTC 概率、损失、解码结果或训练证据
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_root()
BLANK = "∅"
plt.rcParams["figure.figsize"] = (11, 4)
print("项目根目录:", ROOT)

import soundfile as sf
import librosa
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
torch.manual_seed(3); np.random.seed(3)

## 1. 构造多数字语音

把 0～9 的开源单数字录音随机拼接，中间插入短静音，同时保存字符标签。

In [ ]:
files={str(i):ROOT/"data"/"spoken_digits_parts"/f"{i}_jackson_0.wav" for i in range(10)}
waves={}; sr0=None
for d,p in files.items():
    y,sr=sf.read(p); y=y.astype(np.float32); sr0=sr
    waves[d]=y/(np.max(np.abs(y))+1e-8)

def make_example(text):
    silence=np.zeros(int(sr0*.06),np.float32)
    return np.concatenate([z for i,d in enumerate(text) for z in ([waves[d]] if i==len(text)-1 else [waves[d],silence])])

texts=[str(i) for i in range(10)] + ["12","21","34","43","56","65","78","87","90","09","11","22"]
print("样本数",len(texts),"示例",texts[:5],"采样率",sr0)

## 2. Log-Mel 与 batch

In [ ]:
def feat(text):
    y=make_example(text)
    m=librosa.feature.melspectrogram(y=y,sr=sr0,n_fft=256,win_length=200,hop_length=80,n_mels=24,power=2,center=False)
    return torch.tensor(np.log(m+1e-6).T,dtype=torch.float32)
features=[feat(t) for t in texts]
mean=torch.cat(features).mean(0); std=torch.cat(features).std(0).clamp_min(1e-5)
features=[(x-mean)/std for x in features]
lengths=torch.tensor([len(x) for x in features])
padded=pad_sequence(features,batch_first=True)
targets=torch.tensor([int(c)+1 for t in texts for c in t])
target_lengths=torch.tensor([len(t) for t in texts])
print("features [N,T,F]",padded.shape,"targets",targets.shape)

## 3. 小型双向 GRU + CTC head

双向 GRU 是为了让小实验容易收敛；它不是流式编码器。第 16 课会改为因果结构。

In [ ]:
class TinyCTC(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn=nn.GRU(24,32,batch_first=True,bidirectional=True)
        self.head=nn.Linear(64,11)  # 0 blank, 1..10 digits
    def forward(self,x):
        return self.head(self.rnn(x)[0])

model=TinyCTC(); opt=torch.optim.Adam(model.parameters(),lr=8e-3)
ctc=nn.CTCLoss(blank=0,zero_infinity=True)
loss_curve=[]
for epoch in range(160):
    opt.zero_grad(); logits=model(padded)
    loss=ctc(logits.log_softmax(-1).transpose(0,1),targets,lengths,target_lengths)
    loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5); opt.step()
    loss_curve.append(loss.item())
print("初始/最终 loss:",loss_curve[0],loss_curve[-1])
plt.plot(loss_curve);plt.xlabel("Epoch");plt.ylabel("CTC loss");plt.title("Tiny CTC overfitting experiment");plt.show()

## 4. Greedy 查看训练集结果

In [ ]:
def decode(ids):
    out=[];prev=None
    for x in ids:
        if x!=0 and x!=prev: out.append(str(x-1))
        prev=x
    return "".join(out)
with torch.no_grad(): pred=model(padded).argmax(-1)
for i,t in enumerate(texts): print(f"target={t:>2} pred={decode(pred[i,:lengths[i]].tolist()):>4}")

## 5. 看 blank 如何占据大多数帧

In [ ]:
i=texts.index("12")
with torch.no_grad(): p=model(padded[i:i+1]).softmax(-1)[0,:lengths[i]].numpy().T
plt.imshow(p,aspect="auto",origin="lower",cmap="magma")
plt.yticks(range(11),[BLANK]+list("0123456789"));plt.xlabel("Frame");plt.ylabel("CTC class")
plt.title("CTC posterior for target '12'");plt.colorbar(label="Probability");plt.show()

## 本课测试

1. 这里为什么可以说是“过拟合实验”？
2. 为什么 target 的数字 id 要从 1 开始？
3. 双向 GRU 为什么不能严格流式？
4. 训练后 blank 占很多帧是否必然表示模型失败？
5. 真正评估泛化还缺少什么？

<details><summary>展开参考答案</summary>

1. 训练和展示使用同一小批样本。2. id=0 留给 blank。3. 当前输出依赖未来帧。4. 不一定，这是 CTC 常见的尖峰输出。5. 独立训练/验证/测试划分、更多说话人和 CER/WER。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 14 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `真实音频训练`、`CTC spike`、`过拟合实验`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**训练集全对但测试说话人失败**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**加入独立 speaker split 并计算 CER**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**说明双向 GRU 为什么阻碍严格流式**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：真实音频训练、CTC spike、过拟合实验。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 真实音频训练、CTC spike、过拟合实验。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
